In [1]:
import numpy as np

from lab.sinus_vs_square_hard.data import create_data
from quantum_learn.build_f import BuildFQuadratures, BuildFQuadraturesPolynomials, BuildFPhotonDistribution
from quantum_simulation.configs import jpc_config_dudas, quantum_parameters_dudas, base_encoding
from quantum_simulation.history import SimulationResult
from quantum_simulation.simulator import Simulator

In [2]:
def J(F: np.ndarray, Y: np.ndarray) -> float:
    """
    Calcule le score de séparabilité de Fisher pour des données multidimensionnelles.
    F : Matrice (n_samples, n_features)
    Y : Vecteur de labels (0 ou 1)
    """

    class1 = F[Y == 1, :]
    class2 = F[Y == 0, :]

    print(class1.shape)
    print(class2.shape)

    m1 = np.mean(class1, axis=0)
    m2 = np.mean(class2, axis=0)

    s1 = np.cov(class1, rowvar=False)
    s2 = np.cov(class2, rowvar=False)
    Sw = s1 + s2

    diff = m1 - m2

    inv_Sw_diff = np.linalg.solve(Sw, diff)
    score = diff.dot(inv_Sw_diff)

    return float(score)

In [3]:
nb_periods_train = 20

X_train_base, Y_train_base = create_data(nb_periods_train)
X_train, Y_train = X_train_base.reshape(-1), Y_train_base.reshape(-1)

X_train.shape, Y_train.shape

((160,), (160,))

In [4]:
jpc_config_dudas.summary()
quantum_parameters_dudas.summary()

simulator = Simulator(jpc_config_dudas, base_encoding)
state_history: SimulationResult = simulator(X_train, [quantum_parameters_dudas])[0]


JPCConfig(
    DIM_A=10,
    DIM_B=10,
    OMEGA_A=10000.0,
    OMEGA_B=9000.0,
    KAPPA_A=17,
    KAPPA_B=21,
    K_AA=0.1,
    K_BB=0.1,
    K_AB=0.05,
    DRIVE_DURATION=0.04,
    MEASURE_RESOLUTION=10,
    SIMULATION_RESOLUTION=100
)
QuantumParameters(
    g_conv=900,
    g_sq=180,
    epsilon_a=170,
    epsilon_b=170,
    delta_a=0,
    delta_b=0
)


|██████████| 100.0% ◆ elapsed 03m24s ◆ remaining 0.00ms  
ERROR:2026-04-16 16:01:43,780:jax._src.callback:95: jax.pure_callback failed
Traceback (most recent call last):
  File "C:\Users\ozher\AppData\Local\Programs\Python\Python312\Lib\site-packages\jax\_src\callback.py", line 93, in pure_callback_impl
    return tree_util.tree_map(np.asarray, callback(*args))
                                          ^^^^^^^^^^^^^^^
  File "C:\Users\ozher\AppData\Local\Programs\Python\Python312\Lib\site-packages\jax\_src\callback.py", line 71, in __call__
    return tree_util.tree_leaves(self.callback_func(*args, **kwargs))
                                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\ozher\AppData\Local\Programs\Python\Python312\Lib\site-packages\equinox\_errors.py", line 88, in raises
    raise _EquinoxRuntimeError(
equinox._errors._EquinoxRuntimeError: The maximum number of solver steps was reached. Try increasing `max_steps`.


--------------------
An error occurred during

XlaRuntimeError: INTERNAL: CpuCallback error calling callback: Traceback (most recent call last):
  File "C:\Users\ozher\AppData\Local\Programs\Python\Python312\Lib\site-packages\jax\_src\callback.py", line 782, in _wrapped_callback
  File "C:\Users\ozher\AppData\Local\Programs\Python\Python312\Lib\site-packages\jax\_src\callback.py", line 222, in _callback
  File "C:\Users\ozher\AppData\Local\Programs\Python\Python312\Lib\site-packages\jax\_src\callback.py", line 96, in pure_callback_impl
  File "C:\Users\ozher\AppData\Local\Programs\Python\Python312\Lib\site-packages\jax\_src\callback.py", line 71, in __call__
  File "C:\Users\ozher\AppData\Local\Programs\Python\Python312\Lib\site-packages\equinox\_errors.py", line 88, in raises
_EquinoxRuntimeError: The maximum number of solver steps was reached. Try increasing `max_steps`.


--------------------
An error occurred during the runtime of your JAX program! Unfortunately you do not appear to be using `equinox.filter_jit` (perhaps you ar

In [ ]:
build_f_quadratures = BuildFQuadratures(jpc_config_dudas)
build_f_quadratures_polynomials = BuildFQuadraturesPolynomials(jpc_config_dudas)
build_f_photon_distribution = BuildFPhotonDistribution(jpc_config_dudas, clip_probas=2)

F_quad = build_f_quadratures(state_history)
F_quad_poly = build_f_quadratures_polynomials(state_history)
F_photon = build_f_photon_distribution(state_history)

print(F_quad.shape)

In [ ]:
print(J(F_quad, Y_train))
print(J(F_quad_poly, Y_train))

In [ ]:
from zeroth.all import *

X_train_quad = F_quad
X_train_poly = F_quad_poly
X_train_photon = F_photon
Y_train_data = Y_train[:, None]

X_train_quad.shape, X_train_poly.shape, X_train_photon.shape, Y_train_data.shape

In [ ]:
data_quad = Data(X_train_quad, Y_train_data, X_train_quad, Y_train_data, 2)
data_poly = Data(X_train_poly, Y_train_data, X_train_poly, Y_train_data, 2)
data_photon = Data(X_train_photon, Y_train_data, X_train_photon, Y_train_data, 2)

In [ ]:
nn = NeuralNetworkConfig(name="linear", hidden_dims=[], activations=[Softmax()])
optimizer = FirstOrderAdamConfig(learning_rate=0.05,
                                 beta1=0.9,
                                 beta2=0.99,
                                 epsilon=1e-8)
model_config = FirstOrderModelConfig(name="linear",
                                     id={},
                                     loss=CrossEntropy(),
                                     metric=Accuracy(),
                                     batch_size=nb_periods_train,
                                     nb_epochs=100,
                                     neural_network_config=nn,
                                     optimizer_config=optimizer)
model_config.summary()

In [ ]:
model_quad = model_config.instantiate(data_quad)
model_quad.name = "model_quad"

model_poly = model_config.instantiate(data_poly)
model_poly.name = "model_poly"

model_photon = model_config.instantiate(data_photon)
model_photon.name = "model_photon"

models = [model_quad, model_poly, model_photon]

In [ ]:
for model in models:
    model.train()

In [ ]:
model_quad.plot_loss(0.05)

In [ ]:
model_poly.plot_loss(0.05)

In [ ]:
model_photon.plot_loss(0.05)

In [ ]:
for model in models:
    model.test()

In [ ]:
model_poly.plot_loss(0.05)